### 23L-2620


In [7]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
import nltk
from nltk.stem import PorterStemmer
from collections import Counter
from nltk.corpus import stopwords

In [8]:
# Check NLTK data safely
have_nltk_data = True
try:
    nltk.data.find('tokenizers/punkt')
    nltk.data.find('corpora/stopwords')
except LookupError:
    have_nltk_data = False

ps = PorterStemmer()

df = pd.read_csv("spam.csv", encoding="latin-1")  # Load dataset

if "v1" in df.columns and "v2" in df.columns:
    df = df[["v1", "v2"]].rename(columns={"v1": "label", "v2": "message"})
elif "label" in df.columns and "message" in df.columns:
    df = df[["label", "message"]]
else:
    df = df.iloc[:, :2]
    df.columns = ['label','message']

df = df.dropna().reset_index(drop=True)
df['label_num'] = df['label'].apply(lambda x: 1 if str(x).strip().lower()=='spam' else 0)

# Stopwords
if have_nltk_data:
    try:
        stop_words = set(stopwords.words('english'))
    except Exception:
        stop_words = set(["the","is","in","and","to","a","of","that","it","on","for","with","as","this","are","was"])
else:
    stop_words = set(["the","is","in","and","to","a","of","that","it","on","for","with","as","this","are","was"])

def preprocess_text(text):
    text = str(text).lower()
    
    # Replace punctuation with spaces using string operations
    punctuation = '''!()-[]{};:'"\,<>./?@#$%^&*_~'''
    for char in punctuation:
        text = text.replace(char, ' ')
    
    try:
        tokens = nltk.word_tokenize(text)
    except Exception:
        # Simple word splitting if NLTK fails
        tokens = text.split()
    
    tokens = [ps.stem(t) for t in tokens if t not in stop_words and len(t) > 1]
    return tokens

df['tokens'] = df['message'].apply(preprocess_text)
df['processed'] = df['tokens'].apply(lambda toks: ' '.join(toks))
df.head()

<>:37: SyntaxWarning: invalid escape sequence '\,'
<>:37: SyntaxWarning: invalid escape sequence '\,'
C:\Users\Admin\AppData\Local\Temp\ipykernel_17316\610260640.py:37: SyntaxWarning: invalid escape sequence '\,'
  punctuation = '''!()-[]{};:'"\,<>./?@#$%^&*_~'''


,label,message,label_num,tokens,processed
0,ham,"Go until jurong point, crazy.. Available only ...",0,"[go, until, jurong, point, crazi, avail, onli,...",go until jurong point crazi avail onli bugi gr...
1,ham,Ok lar... Joking wif u oni...,0,"[ok, lar, joke, wif, oni]",ok lar joke wif oni
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1,"[free, entri, wkli, comp, win, fa, cup, final,...",free entri wkli comp win fa cup final tkt 21st...
3,ham,U dun say so early hor... U c already then say...,0,"[dun, say, so, earli, hor, alreadi, then, say]",dun say so earli hor alreadi then say
4,ham,"Nah I don't think he goes to usf, he lives aro...",0,"[nah, don, think, he, goe, usf, he, live, arou...",nah don think he goe usf he live around here t...


In [9]:
# N-gram counts
def build_ngram_counts(token_lists, n=1):
    counts = Counter()
    total = 0
    for tokens in token_lists:
        if n == 1:
            for w in tokens:
                counts[(w,)] += 1
                total += 1
        else:
            for i in range(len(tokens) - n + 1):
                gram = tuple(tokens[i : i + n])
                counts[gram] += 1
                total += 1
    return counts, total


def vocab_from_counts(unigram_counts):
    return set(w for (w,) in unigram_counts.keys())


unigram_counts, unigram_total = build_ngram_counts(df["tokens"], n=1)
bigram_counts, bigram_total = build_ngram_counts(df["tokens"], n=2)
V = len(vocab_from_counts(unigram_counts))


def unigram_prob(word, unigram_counts, total, V, alpha=1.0):
    return (unigram_counts.get((word,), 0) + alpha) / (total + alpha * V)


def bigram_prob(prev, word, bigram_counts, unigram_counts, V, alpha=1.0):
    prev_count = unigram_counts.get((prev,), 0)
    num = bigram_counts.get((prev, word), 0) + alpha
    den = prev_count + alpha * V
    return num / den if den > 0 else 1.0 / V


def sentence_unigram_logprob(tokens, unigram_counts, total, V, alpha=1.0):
    lp = 0.0
    for w in tokens:
        p = unigram_prob(w, unigram_counts, total, V, alpha)
        lp += np.log(p)
    return lp


def sentence_bigram_logprob(tokens, bigram_counts, unigram_counts, V, alpha=1.0):
    lp = 0.0
    for i in range(1, len(tokens)):
        prev = tokens[i - 1]
        w = tokens[i]
        p = bigram_prob(prev, w, bigram_counts, unigram_counts, V, alpha)
        lp += np.log(p)
    if len(tokens) > 0:
        lp += np.log(unigram_prob(tokens[0], unigram_counts, unigram_total, V, alpha))
    return lp


def perplexity_from_logprob(logprob, N):
    return np.exp(-logprob / N) if N > 0 else float("inf")


sample_sent = df["tokens"].iloc[0]
uni_lp = sentence_unigram_logprob(sample_sent, unigram_counts, unigram_total, V)
bi_lp = sentence_bigram_logprob(sample_sent, bigram_counts, unigram_counts, V)
uni_perp = perplexity_from_logprob(uni_lp, len(sample_sent))
bi_perp = perplexity_from_logprob(bi_lp, len(sample_sent))
uni_lp, bi_lp, uni_perp, bi_perp

(np.float64(-134.1944553585641),
 np.float64(-136.7184283821991),
 np.float64(2680.588105228038),
 np.float64(3109.6345588327677))

In [10]:
# Naive Bayes Classification
X = df['processed']
y = df['label_num']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

vectorizer = CountVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

clf = MultinomialNB()
clf.fit(X_train_vec, y_train)

y_pred = clf.predict(X_test_vec)
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)

Accuracy: 0.9838565022421525
Precision: 0.9712230215827338
Recall: 0.9060402684563759
F1: 0.9375


In [11]:
# Laplace smoothing experiment
alphas = [1.0, 0.5, 0.1]
for a in alphas:
    m = MultinomialNB(alpha=a)
    m.fit(X_train_vec, y_train)
    yp = m.predict(X_test_vec)
    print(
        f"Alpha={a}:",
        "Precision=",
        precision_score(y_test, yp, zero_division=0),
        "Recall=",
        recall_score(y_test, yp, zero_division=0),
        "F1=",
        f1_score(y_test, yp, zero_division=0),
        "Accuracy=",
        accuracy_score(y_test, yp),
    )

Alpha=1.0: Precision= 0.9712230215827338 Recall= 0.9060402684563759 F1= 0.9375 Accuracy= 0.9838565022421525
Alpha=0.5: Precision= 0.971830985915493 Recall= 0.9261744966442953 F1= 0.9484536082474226 Accuracy= 0.9865470852017937
Alpha=0.1: Precision= 0.9583333333333334 Recall= 0.9261744966442953 F1= 0.9419795221843004 Accuracy= 0.9847533632286996
